En este archivo Notebook desarrollo parte por parte el código Python del proyecto para su análisis de funcionamiento e incorporación a la memoria.

In [25]:
# Añadimos todas las librerias necesarias
import getpass
import oracledb as oracledb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [35]:

#Importamos los datos en csv
df = pd.read_csv("C:/Users/Jesús/Desktop/sqldeveloper/sqldeveloper/bin/export.csv")
df.head()


,CENTRO,TITULACIÓN,CURSOACADÉMICO,CODIGOALUM,CÓDIGOASIGNATURAUMA,NOMBREASIGNATURA,SEMESTRE,CALIFICACIÓN,NUM_CALIFICACIÓN,CONVOCATORIAS
0,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA INFORMÁTICA POR LA UN...,2018-19,0208F18506E41D3F29A4CAAD842FD0FA,50661,Fundamentos de la Programación,1,APROBADO,5,Segunda convocatoria ordinaria
1,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA INFORMÁTICA POR LA UN...,2018-19,0208F18506E41D3F29A4CAAD842FD0FA,50661,Fundamentos de la Programación,1,SUSPENSO,3,Primera convocatoria ordinaria
2,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA INFORMÁTICA POR LA UN...,2018-19,0208F18506E41D3F29A4CAAD842FD0FA,50662,Matemática Discreta,1,SUSPENSO,1,Primera convocatoria ordinaria
3,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA INFORMÁTICA POR LA UN...,2018-19,0208F18506E41D3F29A4CAAD842FD0FA,50662,Matemática Discreta,1,SUSPENSO,1,Segunda convocatoria ordinaria
4,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA INFORMÁTICA POR LA UN...,2018-19,0208F18506E41D3F29A4CAAD842FD0FA,50663,Estructuras Algebraicas para la Computación,2,NO PRESENTADO,0,Primera convocatoria ordinaria


Cálculo de posibilidad de entrada a asignatura: Considero la posibilidad de entrar a la asignatura como el porcentaje de cursos en los que la nota de corte es inferior a la nota media del alumno.

In [40]:
#TODO: La nota media no considera ni suspensos ni no presentados
#Aplico filtrado por nombre de asignatura
filtro = df["NOMBREASIGNATURA"] == "Programación de Videojuegos"
df_filtro_asignatura = df[filtro]

# Obtener los alumnos que cursaron esa asignatura
alumnos_objetivo = df_filtro_asignatura['CODIGOALUM'].unique()


# Filtrar solo registros de esos alumnos
df_alumnos = df[df['CODIGOALUM'].isin(alumnos_objetivo)].copy()
df_alumnos

,CENTRO,TITULACIÓN,CURSOACADÉMICO,CODIGOALUM,CÓDIGOASIGNATURAUMA,NOMBREASIGNATURA,SEMESTRE,CALIFICACIÓN,NUM_CALIFICACIÓN,CONVOCATORIAS,AÑO_INICIAL
154,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DE COMPUTADORES POR L...,2018-19,98EB0889B4743C7FA4846844AADFCBD4,51673,Cálculo para la Computación,1,NO PRESENTADO,0,Primera convocatoria ordinaria,2018
155,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DE COMPUTADORES POR L...,2018-19,98EB0889B4743C7FA4846844AADFCBD4,51674,Fundamentos Físicos de la Informática,1,NO PRESENTADO,0,Primera convocatoria ordinaria,2018
156,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DE COMPUTADORES POR L...,2018-19,98EB0889B4743C7FA4846844AADFCBD4,51674,Fundamentos Físicos de la Informática,1,NO PRESENTADO,0,Segunda convocatoria ordinaria,2018
157,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DE COMPUTADORES POR L...,2018-19,98EB0889B4743C7FA4846844AADFCBD4,51675,Fundamentos de Electrónica,1,SUSPENSO,1,Primera convocatoria ordinaria,2018
158,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DE COMPUTADORES POR L...,2018-19,98EB0889B4743C7FA4846844AADFCBD4,51676,Fundamentos de la Programación,1,NO PRESENTADO,0,Segunda convocatoria ordinaria,2018
...,...,...,...,...,...,...,...,...,...,...,...
96626,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,MÁSTER UNIVERSITARIO EN INGENIERÍA INFORMÁTICA...,2022-23,C00ED1AC75E39B3F4204E2EA5A98AC99,102689,Análisis Visual de Datos (Visual Data Analysis),1,MATRÍCULA HONOR,9,Primera convocatoria ordinaria,2022
96811,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DEL SOFTWARE POR LA U...,2022-23,B73909AF13C8050CDFB1BB3036933969,51180,Gestión de Proyectos Software,1,MATRÍCULA HONOR,9,Primera convocatoria ordinaria,2022
96812,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DEL SOFTWARE POR LA U...,2022-23,B73909AF13C8050CDFB1BB3036933969,51182,Métodos formales para la Ingeniería del Software,1,MATRÍCULA HONOR,9,Primera convocatoria ordinaria,2022
96813,ESCUELA TÉCNICA SUPERIOR DE INGENIERÍA INFORMÁ...,GRADUADO/A EN INGENIERÍA DEL SOFTWARE POR LA U...,2022-23,B73909AF13C8050CDFB1BB3036933969,51196,Prácticas Externas,2,MATRÍCULA HONOR,10,Primera convocatoria ordinaria,2022


In [5]:
# Convertir CURSOACADÉMICO a año inicial para comparar
df_alumnos['AÑO_INICIAL'] = df_alumnos['CURSOACADÉMICO'].str[:4].astype(int)
df_filtro_asignatura['AÑO_INICIAL'] = df_filtro_asignatura['CURSOACADÉMICO'].str[:4].astype(int)

# Unir para saber en qué año cursaron la asignatura objetivo
df_merge = df_filtro_asignatura[['CODIGOALUM', 'AÑO_INICIAL']].rename(columns={'AÑO_INICIAL': 'AÑO_OBJETIVO'})

# Combinar con todos sus datos para filtrar por año
df_con_contexto = pd.merge(df_alumnos, df_merge, on='CODIGOALUM')

# Filtrar solo asignaturas cursadas **antes** del año de la asignatura objetivo
df_anteriores = df_con_contexto[df_con_contexto['AÑO_INICIAL'] < df_con_contexto['AÑO_OBJETIVO']]

# Calcular la media de notas por alumno
promedios = df_anteriores.groupby('CODIGOALUM')['NUM_CALIFICACIÓN'].mean().reset_index()
promedios.rename(columns={'NUM_CALIFICACIÓN': 'MEDIA_ANTERIOR'}, inplace=True)
promedios

C:\Users\Jesús\AppData\Local\Temp\ipykernel_12208\4002167851.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtro_asignatura['AÑO_INICIAL'] = df_filtro_asignatura['CURSOACADÉMICO'].str[:4].astype(int)


,CODIGOALUM,MEDIA_ANTERIOR
0,003F9854C10615AADCF46D3CE4B6F0AA,3.902439
1,0569333B0A216277737205515DDE4E49,5.333333
2,07FB91FACB9F88DE531682C7874D0E50,6.000000
3,09D3DA93C1195F9337C98092057E86BE,2.363636
4,0C7267A8DEF33A0DE76B47684B852210,3.425532
...,...,...
126,F7D9235CF8E82B3E9E832849962D71A0,4.038462
127,F9EE4394854A8F0688F82D8D850DE2BA,4.592593
128,FB5A5EFEBD0FC95862D77C532C9BE811,5.833333
129,FCC79D7FBA9C8876CBDB8FA4EA0BA236,1.777778


In [33]:
#Para la asignatura, obtengo todos los alumnos que la han cursado
asignatura = "Gestión Inteligente de la Información"
df_asignatura = df[
    (df['NOMBREASIGNATURA'] == asignatura) 
]

#Obtengo todos los alumnos de la base de datos que se han matriculado en la asignatura
alumnos_objetivo = df_filtro_asignatura['CODIGOALUM'].unique()
#print(alumnos_objetivo)

# Asegurarse de que el campo CURSOACADÉMICO está bien tratado
df['AÑO_INICIAL'] = df['CURSOACADÉMICO'].str[:4].astype(int)
df_asignatura['AÑO_INICIAL'] = df_asignatura['CURSOACADÉMICO'].str[:4].astype(int)

# Lista para guardar resultados
resultados = []

# Recorremos cada alumno que ha cursado la asignatura
for alumno in alumnos_objetivo:
    # Obtener todos los años en los que el alumno cursó la asignatura
    años_matricula = df_asignatura[df_asignatura['CODIGOALUM'] == alumno]['AÑO_INICIAL'].unique()
    
    # Para cada matrícula del alumno en esa asignatura (por si repitió)
    for año_matricula in años_matricula:
        # Filtrar asignaturas cursadas por el alumno ANTES de ese año
        historial = df[
            (df['CODIGOALUM'] == alumno) &
            (df['AÑO_INICIAL'] < año_matricula) &
            (df['CALIFICACIÓN'].str.lower() != "no presentado") &
            (df['NUM_CALIFICACIÓN'] >= 5)  # Solo aprobados
        ]
        
        if len(historial) > 0:
            media = historial['NUM_CALIFICACIÓN'].mean()
        else:
            media = None  # O podrías usar NaN
        
        resultados.append({
            'CODIGOALUM': alumno,
            'AÑO_MATRICULA': año_matricula,
            'MEDIA_ANTERIOR': media
        })

# Crear DataFrame final
df_resultado = pd.DataFrame(resultados)
df_resultado = df_resultado.dropna(subset=['MEDIA_ANTERIOR'])
print(df_resultado)

C:\Users\Jesús\AppData\Local\Temp\ipykernel_12208\733321488.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_asignatura['AÑO_INICIAL'] = df_asignatura['CURSOACADÉMICO'].str[:4].astype(int)


                           CODIGOALUM  AÑO_MATRICULA  MEDIA_ANTERIOR
16   D28472CF8C6886F7CBB47C0E2FEDC400           2019        7.285714
17   FB5A5EFEBD0FC95862D77C532C9BE811           2019        6.090909
18   C3A52996CB5770A67FD5DB516D9F91EA           2019        6.300000
19   328DFF715F1AF124A64B2F69462E8883           2019        6.375000
20   178D63EC03109C0ED4EA23DED6674F89           2019        7.625000
..                                ...            ...             ...
146  3886E9D8475C30F264BAFBA7EBAB9F0A           2022        6.650000
147  AACA0E4685A6A3B516C207AC2C5355CF           2022        6.360000
148  B67813AF7B97934DC3348AC544262422           2022        6.772727
149  8BD1E188326A767426BFA2EAC0A07757           2022        6.120000
150  DA4793EB2857BBCD7C7DE8D4EFDDDF47           2022        6.600000

[133 rows x 3 columns]


In [42]:
asignatura = "Laboratorio de Computación Científica" 

# Filtrar por la asignatura
df_asignatura = df[df['NOMBREASIGNATURA'] == asignatura].copy()

# Asegurar tratamiento del año
df['AÑO_INICIAL'] = df['CURSOACADÉMICO'].str[:4].astype(int)
df_asignatura['AÑO_INICIAL'] = df_asignatura['CURSOACADÉMICO'].str[:4].astype(int)

# Obtener todos los años en los que se impartió la asignatura
años_matricula = df_asignatura['AÑO_INICIAL'].unique()

# Calcular la NOTA_CORTE para cada año (mínima media anterior entre los alumnos de ese año)
nota_corte_anual = []

for año in años_matricula:
    alumnos_ano = df_asignatura[df_asignatura['AÑO_INICIAL'] == año]['CODIGOALUM'].unique()
    medias = []

    for alumno in alumnos_ano:
        historial = df[
            (df['CODIGOALUM'] == alumno) &
            (df['AÑO_INICIAL'] < año) &
            (df['CALIFICACIÓN'].str.lower() != "no presentado") &
            (df['NUM_CALIFICACIÓN'] >= 5)
        ]
        if not historial.empty:
            media = historial['NUM_CALIFICACIÓN'].mean()
            medias.append(media)
    
    if medias:
        nota_corte_anual.append({
            'AÑO_MATRICULA': año,
            'NOTA_CORTE': min(medias)
        })

df_nota_corte = pd.DataFrame(nota_corte_anual)

# -----------------------------------------
# Ahora para un alumno específico:
alumno_objetivo = "FB5A5EFEBD0FC95862D77C532C9BE811"

# Calcular su media anterior para cada año en que se impartió la asignatura
medias_objetivo = []
for año in df_nota_corte['AÑO_MATRICULA']:
    historial = df[
        (df['CODIGOALUM'] == alumno_objetivo) &
        (df['AÑO_INICIAL'] < año) &
        (df['CALIFICACIÓN'].str.lower() != "no presentado") &
        (df['NUM_CALIFICACIÓN'] >= 5)
    ]
    if not historial.empty:
        media = historial['NUM_CALIFICACIÓN'].mean()
        medias_objetivo.append({
            'AÑO_MATRICULA': año,
            'MEDIA_OBJETIVO': media
        })

df_media_obj = pd.DataFrame(medias_objetivo)

# -----------------------------------------
# Unimos ambos DF para comparar
df_comparacion = pd.merge(df_nota_corte, df_media_obj, on='AÑO_MATRICULA')

media_alumno = 4.56

# Añadir columna indicando si el alumno habría podido entrar
df_comparacion['ENTRA'] = df_media_obj['MEDIA_OBJETIVO'] >= df_comparacion['NOTA_CORTE']

# Calcular probabilidad de entrada
probabilidad_entrada = df_comparacion['ENTRA'].mean()

print("Comparación año por año:")
print(df_comparacion)

print(f"\nProbabilidad de entrada del alumno {alumno_objetivo} en '{asignatura}': {probabilidad_entrada:.2%}")

Comparación año por año:
   AÑO_MATRICULA  NOTA_CORTE  MEDIA_OBJETIVO  ENTRA
0           2019    5.000000        6.090909   True
1           2020    5.615385        6.300000   True
2           2021    5.500000        6.300000   True
3           2022    5.833333        6.300000   True

Probabilidad de entrada del alumno FB5A5EFEBD0FC95862D77C532C9BE811 en 'Laboratorio de Computación Científica': 100.00%
